In [ ]:
!pip install -q langchain langchain-ollama

In [ ]:
# loading API key
from google.colab import userdata

ollama_api_key = userdata.get("OLLAMA_API_KEY")

print("API  key Found : ",bool(ollama_api_key))

API  key Found :  True


In [ ]:
# model creation
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {ollama_api_key}"
        }
    },
    temperature=0.1
)

In [ ]:
# testing Model
print(llm.invoke("which model you are?").content)

I’m ChatGPT, powered by OpenAI’s GPT‑4 Turbo architecture. This is the latest generation of the GPT‑4 family, optimized for speed, cost‑efficiency, and conversational quality. If you have any specific questions about my capabilities or how I work, feel free to ask!


In [33]:
# tool creation : Discount calculator
from langchain_core.tools import tool
@tool
def discount_calculator(price :float , discount_rate: float)->dict:
  """Calculate the Discount and final price"""
  discount = price  * discount_rate / 100
  final_price = price - discount
  return{
      "Price": price,
      "Discount_rate":discount_rate,
      "Discount":discount,
      "Final Price ": final_price
  }

# testing the tool
result = discount_calculator.invoke({"price":1800,"discount_rate":10})
print(result)

{'Price': 1800.0, 'Discount_rate': 10.0, 'Discount': 180.0, 'Final Price ': 1620.0}


In [38]:
# tool creation : GST calculator
@tool
def gst_calculator(amount :float , gst_rate : float )-> dict:
  """calculate the GST and Final Amount"""

  gst = amount * gst_rate / 100
  final_amount = amount + gst
  return{
      "Amount": amount,
      "GST Rate": gst_rate,
      "GST" : gst,
      "Fianl Amount": final_amount
  }

# testing the tool
result = gst_calculator.invoke({ "amount" : 1200, "gst_rate" : 28})
print(result)



{'Amount': 1200.0, 'GST Rate': 28.0, 'GST': 336.0, 'Fianl Amount': 1536.0}


In [39]:
# creating Agent
from langchain.agents import create_agent
agent = create_agent(
    model = llm,
    tools = [discount_calculator,gst_calculator]
)


In [40]:
# user Query
user_question = "the price of product is 800000 and the discount rate is 25 percent and the amount of car is 1200000 and gst rate is 28 percent"

In [41]:
# agent calling
result = agent.invoke({
    "messages" : [
        {
            "role":"user",
            "content":user_question
        }
    ]
})

In [42]:
#result
print(result["messages"][-1].content)

Here are the calculations based on the figures you provided:

| Item | Value | Calculation | Result |
|------|-------|-------------|--------|
| **Product price** | ₹800,000 |  | ₹800,000 |
| **Discount rate** | 25 % | 800,000 × 0.25 | ₹200,000 |
| **Discount amount** |  |  | **₹200,000** |
| **Final price after discount** |  | 800,000 – 200,000 | **₹600,000** |
| **Car amount** | ₹1,200,000 |  | ₹1,200,000 |
| **GST rate** | 28 % | 1,200,000 × 0.28 | ₹336,000 |
| **GST amount** |  |  | **₹336,000** |
| **Final amount after GST** |  | 1,200,000 + 336,000 | **₹1,536,000** |

**Summary**

- **Product**: ₹600,000 after a 25 % discount.  
- **Car**: ₹1,536,000 after adding 28 % GST to the base amount of ₹1,200,000.  

Let me know if you need any further breakdown or additional calculations!
